# Crawling Data from `vnstock` library

## Libraries

In [1]:
import os
import time
import random
import traceback
import pandas as pd
from vnstock import Quote

## Configuration and Helper Functions

In [2]:
# --------------------------------------------------
# Config
# --------------------------------------------------
home = "./data"
os.makedirs(home, exist_ok=True)

symbols = [
    # Banking
    "VCB","BID","CTG","TCB","MBB","ACB","VPB","STB","HDB",
    "VIB","TPB","LPB","SHB",

    # Real estate
    "VHM","VIC","VRE",

    # Industry
    "HPG","HSG", "DGC","GVR",

    # Consumer
    "MWG","MSN","VNM","SAB","PNJ",

    # Energy
    "POW","GAS","PLX",

    # Others
    "FPT","SSI","VJC","BVH"
]

start = "2018-01-01"
end   = "2026-03-15"

# KBS is usually easier on Colab/Kaggle; 
# VCI can be more complete on local machines
source = "VCI"

In [3]:
def normalize_history(df: pd.DataFrame, symbol: str) -> pd.DataFrame:
    """
    Standardize vnstock history output to:
    date, symbol, open, high, low, close, volume
    """
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=["date", "symbol", "open", "high", "low", "close", "volume"])

    out = df.copy()

    out = out.rename(columns={"time": "date"})

    required_price_cols = ["open", "high", "low", "close", "volume"]
    missing = [c for c in required_price_cols if c not in out.columns]
    if missing:
        raise ValueError(f"{symbol}: missing expected columns {missing}. Got: {list(out.columns)}")

    out["date"] = pd.to_datetime(out["date"]).dt.tz_localize(None)
    out["symbol"] = symbol

    out = out[["date", "symbol", "open", "high", "low", "close", "volume"]].copy()
    out = out.sort_values("date").drop_duplicates(subset=["date"]).reset_index(drop=True)
    return out


def download_one_symbol(symbol: str, start: str, end: str, source: str) -> pd.DataFrame:
    """
    Download one symbol from vnstock.
    """
    q = Quote(symbol=symbol, source=source)
    df = q.history(start=start, end=end, interval="1D")
    return normalize_history(df, symbol)


def save_symbol_csv(df: pd.DataFrame, folder: str, symbol: str):
    path = os.path.join(folder, f"{symbol}.csv")
    df.to_csv(path, index=False)
    print(f"Saved {symbol}: {len(df):,} rows -> {path}")


## Downloading

In [11]:
all_frames = []
failed = []

for i, symbol in enumerate(symbols, 1):
    try:
        print(f"[{i}/{len(symbols)}] Downloading {symbol} from {source} ...")
        df_sym = download_one_symbol(symbol, start=start, end=end, source=source)

        if df_sym.empty:
            print(f"  -> Empty data for {symbol}")
            failed.append((symbol, "empty dataframe"))
            continue

        save_symbol_csv(df_sym, home, symbol)
        all_frames.append(df_sym)

        # be polite to source
        time.sleep(0.8 + random.random() * 0.8)

    except Exception as e:
        print(f"  -> Failed {symbol}: {e}")
        failed.append((symbol, str(e)))
        # optional debug:
        # traceback.print_exc()
        time.sleep(1.5)

[1/14] Downloading DGC from VCI ...
Saved DGC: 2,142 rows -> ./data\DGC.csv
[2/14] Downloading GVR from VCI ...
Saved GVR: 1,987 rows -> ./data\GVR.csv
[3/14] Downloading MWG from VCI ...
Saved MWG: 2,142 rows -> ./data\MWG.csv
[4/14] Downloading MSN from VCI ...
Saved MSN: 2,142 rows -> ./data\MSN.csv
[5/14] Downloading VNM from VCI ...
Saved VNM: 2,142 rows -> ./data\VNM.csv
[6/14] Downloading SAB from VCI ...
Saved SAB: 2,142 rows -> ./data\SAB.csv
[7/14] Downloading PNJ from VCI ...
Saved PNJ: 2,142 rows -> ./data\PNJ.csv
[8/14] Downloading POW from VCI ...
Saved POW: 1,995 rows -> ./data\POW.csv
[9/14] Downloading GAS from VCI ...
Saved GAS: 2,142 rows -> ./data\GAS.csv
[10/14] Downloading PLX from VCI ...
Saved PLX: 2,142 rows -> ./data\PLX.csv
[11/14] Downloading FPT from VCI ...
Saved FPT: 2,142 rows -> ./data\FPT.csv
[12/14] Downloading SSI from VCI ...
Saved SSI: 2,142 rows -> ./data\SSI.csv
[13/14] Downloading VJC from VCI ...
Saved VJC: 2,142 rows -> ./data\VJC.csv
[14/14] 

In [13]:
# --------------------------------------------------
# Save failures
# --------------------------------------------------
if failed:
    failed_df = pd.DataFrame(failed, columns=["symbol", "error"])
    print(failed_df)
else:
    print("\nAll symbols downloaded successfully.")


All symbols downloaded successfully.


## Normalize time range

In [4]:
# Filter invalide time range
START = "2018-01-01"
END   = "2026-03-13"

all_frames = []

for file in os.listdir(home):
    path = os.path.join(home, file)
    symbol = file.replace(".csv", "")
    if symbol not in symbols:
        print(f"Skipping unexpected file: {file}")
        continue
    
    df = pd.read_csv(path)
    
    print(f"Processing {symbol} ...")
    print(f"  Original rows: {len(df)}")
    
    # --- filter time range ---
    df = df[(df["date"] >= START) & (df["date"] <= END)]

    # --- basic cleaning ---
    df = df.sort_values("date").drop_duplicates("date")
    
    print(f"  After filtering: {len(df)} rows")
    all_frames.append(df)

Processing ACB ...
  Original rows: 2142
  After filtering: 2039 rows
Processing BID ...
  Original rows: 2142
  After filtering: 2044 rows
Processing BVH ...
  Original rows: 2142
  After filtering: 2044 rows
Skipping unexpected file: coverage_summary.csv
Processing CTG ...
  Original rows: 2142
  After filtering: 2044 rows
Processing DGC ...
  Original rows: 2142
  After filtering: 2038 rows
Processing FPT ...
  Original rows: 2142
  After filtering: 2044 rows
Processing GAS ...
  Original rows: 2142
  After filtering: 2044 rows
Processing GVR ...
  Original rows: 1987
  After filtering: 1987 rows
Processing HDB ...
  Original rows: 2041
  After filtering: 2041 rows
Processing HPG ...
  Original rows: 2142
  After filtering: 2044 rows
Processing HSG ...
  Original rows: 2142
  After filtering: 2044 rows
Processing LPB ...
  Original rows: 2096
  After filtering: 2034 rows
Processing MBB ...
  Original rows: 2142
  After filtering: 2044 rows
Processing MSN ...
  Original rows: 2142
  

## Statistics

In [5]:
# --------------------------------------------------
# Save combined panel and coverage summary
# --------------------------------------------------
if all_frames:
    panel = pd.concat(all_frames, ignore_index=True)
    panel = panel.sort_values(["symbol", "date"]).reset_index(drop=True)

    panel_path = os.path.join(home, "vn30_panel_daily.csv")
    panel.to_csv(panel_path, index=False)
    print(f"\nCombined panel saved: {panel_path}")
    print(panel.head())

    coverage = (
        panel.groupby("symbol")["date"]
        .agg(["min", "max", "count"])
        .sort_values("min")
        .reset_index()
    )
    coverage_path = os.path.join(home, "coverage_summary.csv")
    coverage.to_csv(coverage_path, index=False)

    print("\nCoverage summary:")
    print(coverage)
    print(f"\nCoverage saved: {coverage_path}")
else:
    print("\nNo data downloaded.")


Combined panel saved: ./data\vn30_panel_daily.csv
         date symbol  open  high   low  close   volume
0  2018-01-02    ACB  6.49  6.83  6.46   6.81  3657426
1  2018-01-03    ACB  6.81  6.86  6.67   6.79  5056543
2  2018-01-04    ACB  6.79  6.83  6.74   6.81  6365641
3  2018-01-05    ACB  6.84  6.93  6.74   6.81  6453452
4  2018-01-08    ACB  6.81  7.04  6.81   7.04  3879771

Coverage summary:
   symbol         min         max  count
0     ACB  2018-01-02  2026-03-13   2039
1     VNM  2018-01-02  2026-03-13   2044
2     VJC  2018-01-02  2026-03-13   2044
3     VIC  2018-01-02  2026-03-13   2044
4     VIB  2018-01-02  2026-03-13   2037
5     VCB  2018-01-02  2026-03-13   2044
6     STB  2018-01-02  2026-03-13   2044
7     SSI  2018-01-02  2026-03-13   2044
8     SHB  2018-01-02  2026-03-13   2041
9     SAB  2018-01-02  2026-03-13   2044
10    PNJ  2018-01-02  2026-03-13   2044
11    VPB  2018-01-02  2026-03-13   2044
12    MWG  2018-01-02  2026-03-13   2044
13    PLX  2018-01-02  202